# CASM-256 End-to-End Tutorial

**Goal**: walk through every stage of the CASM cal + beamforming pipeline on
real visibility data, ending with an int8 weight file ready for the online
beamformer.

This notebook reflects the validated production recipe from the 2026-05-15
campaign:
- INACTIVE antenna list = [3] (ant 12 was found to help — Phase 3Q)
- SVDMode.COMPLEX at threshold=4 for cal (better bandpass; phase-only at
  threshold=5 also fine as a safe default)
- Full upper-triangle imaging with |b| > 5 m to avoid cross-talk
- Sun-tracked beam subtraction when Sun is above horizon
- 512 stationary beams written as int8 for the online voltage beamformer

Each section is self-contained. You can change the observation date / source
at the top and the rest will re-flow.

**Prerequisites**: refactor-tower venv (`casm_refactor_env`), data on
`/mnt/nvme*/data/casm/visibilities_64ant/`, May 10 night-static cache.

## 1. Setup

Import what we need from each CASM subpackage. Activate the refactor-tower
venv before launching Jupyter:

```bash
source /home/casm/software/dev/casm_venvs/casm_refactor_env/bin/activate
jupyter notebook
```

In [ ]:
import os
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo

import numpy as np
import matplotlib.pyplot as plt
import h5py

# Core I/O + format
from casm_io.correlator import (
    AntennaMapping, load_format, read_visibilities,
)
from casm_io.correlator.baselines import triu_flat_index, triu_to_ij

# Visibility analysis
from casm_vis_analysis import (
    fringe_stop,
    load_static_visibility, subtract_static_visibility,
    validate_source, plot_source_validation, print_source_validation_summary,
)
from casm_vis_analysis.fringe_stop import compute_baselines_enu
from casm_vis_analysis.sources import source_altaz, source_enu, find_transit_window
from casm_vis_analysis.beam_power import beam_power_vs_time

# Calibrator
from casm_calibrator import (
    svd_calibrate, SVDConfig, SVDMode, save_calibration,
)

# Imaging
from casm_imaging.imaging.beamformer import make_altaz_image

# BF weight generation
from bf_weights_generator import (
    load_calibration_weights,
    Int8StationaryWeights,
)
from bf_weights_generator.io import save_int8_weights_hdf5
from bf_weights_generator.weights import generate_beam_grid_altaz

print("CASM packages loaded OK")

## 2. Observation parameters

Edit this cell to pick a date, source, and time window. The defaults below
target the validated Cyg A pre-sunrise window (May 14, 2026).

In [ ]:
# === USER KNOBS ===
LAYOUT_CSV = "/home/casm/software/dev/antenna_layouts/current"  # symlink to today's layout
DATA_ROOT = "/mnt"                  # where visibilities_64ant/ lives
FORMAT_NAME = "layout_64ant"        # 128 inputs, 137.4 s integration
TZ = "America/Los_Angeles"
REF_ANT = 9                         # cal reference antenna
INACTIVE = [3]                      # validated: keep ant 12 in (Phase 3Q)

# Source-specific time windows (PDT, OVRO local)
SUN_START  = "2026-05-14 12:00:00"
SUN_END    = "2026-05-14 13:30:00"   # Sun transit ±45 min
CYGA_START = "2026-05-14 04:00:00"
CYGA_END   = "2026-05-14 05:30:00"   # Cyg A near transit, Sun below horizon
CASA_START = "2026-05-14 08:05:00"
CASA_END   = "2026-05-14 09:35:00"   # Cas A daytime upper transit

# Static visibility cache (off-source sky model for subtraction)
STATIC_NPZ = "/home/casm/scratch/bf_calibration_march2026/cache/static_2026-05-10_night.npz"

# Output dir for any cals / weights generated by this notebook
OUT_DIR = Path("/tmp/casm_notebook_out")
OUT_DIR.mkdir(exist_ok=True)
(OUT_DIR / "cal").mkdir(exist_ok=True)
(OUT_DIR / "figs").mkdir(exist_ok=True)
(OUT_DIR / "weights").mkdir(exist_ok=True)

print(f"output → {OUT_DIR}")

## 3. Antenna mapping

The layout CSV defines the wiring (which antenna ID lives at which SNAP-board /
ADC input) and the physical ENU positions. `with_inactive([...])` masks bad
antennas so they don't enter cal solves or imaging.

The 21-of-64 active list reflects current CASM-21 deployment. As more
antennas come online, just drop their IDs from `INACTIVE`.

In [ ]:
ant = AntennaMapping.load(LAYOUT_CSV).with_inactive(INACTIVE)
aids = ant.active_antennas()
fmt = load_format(FORMAT_NAME)

print(f"layout       : {LAYOUT_CSV}")
print(f"active ants  : {len(aids)}: {aids}")
print(f"reference    : {REF_ANT}")
print(f"nsig (inputs): {fmt.nsig}")
print(f"nchan        : {fmt.nchan}")
print(f"dt_raw       : {fmt.dt_raw_s} s")

## 4. Read visibilities — Sun transit

`read_visibilities` auto-discovers obs files spanning the requested window.
Returns a `VisibilityResult` with `.vis` `(T, F, n_baseline_full)`, `.freq_mhz`,
`.time_unix`.

We'll use the Sun transit data to solve the cal. Note that `n_baseline_full =
nsig × (nsig+1) / 2 = 8256` is the full upper-triangle including autos — the
21-antenna subset extracts ~210 of those.

In [ ]:
sun_data = read_visibilities(
    time_start=SUN_START, time_end=SUN_END, time_tz=TZ,
    data_root=DATA_ROOT, fmt=fmt, verbose=True,
)
print(f"  vis      : {sun_data.vis.shape}  dtype={sun_data.vis.dtype}  "
      f"({sun_data.vis.nbytes/1e9:.1f} GB)")
print(f"  freq_mhz : [{sun_data.freq_mhz[0]:.2f} → {sun_data.freq_mhz[-1]:.2f}] MHz "
      f"(descending: {sun_data.freq_mhz[0] > sun_data.freq_mhz[-1]})")
print(f"  T_samples: {sun_data.vis.shape[0]}")
print(f"  dt       : {np.median(np.diff(sun_data.time_unix)):.2f} s")

## 5. RFI mask (optional)

CASM at OVRO has a relatively clean 391–484 MHz band, but watch out for:
- MUOS satellite downlink at 360-380 MHz (just below our band, can alias)
- A digital spur at 437.5 MHz (voltage offset between ADCs on SNAP boards)
- Occasional broadband terrestrial RFI

We don't apply an explicit static RFI mask in the validated pipeline — the
SVD cal's rank-1 quality threshold implicitly filters RFI channels by setting
their gain to 0. But you can apply an explicit mask too if you want.

In [ ]:
from casm_vis_analysis.rfi import RFIMask, apply_rfi_mask

# Skip the static RFI mask for now — version 2 was found to zero-out our band.
# If a future version matches, uncomment:
# apply_rfi_mask(sun_data, RFIMask.from_static(version=2))

print("RFI mask: not applied (relying on SVD cal-flag filter downstream)")

## 6. Autocorrelation power spectra

Each antenna's autocorrelation `V_ii` measures its received power per channel
per integration. Sanity checks:
- All antennas should have similar mean power (within a factor of ~3)
- Any antenna at <0.15× of array median is anemic (e.g., ant 12 — Phase 3O)
- The bandpass shape should be roughly flat-ish ±3–6 dB across 391–484 MHz

In [ ]:
pkts = np.array([ant.packet_index(a) for a in aids])
nsig = int(fmt.nsig)

# Extract autocorrelations: |V_ii(t, f)| averaged in time → spectrum per ant
fig, ax = plt.subplots(figsize=(11, 5))
auto_power_median = {}
for aid, pkt in zip(aids, pkts):
    bl_idx = triu_flat_index(nsig, pkt, pkt)
    v_ii = sun_data.vis[:, :, bl_idx]   # (T, F)
    spectrum = np.median(np.abs(v_ii), axis=0)
    ax.plot(sun_data.freq_mhz, 10 * np.log10(spectrum + 1e-30),
            linewidth=0.5, alpha=0.7, label=f"ant {aid}")
    auto_power_median[aid] = float(np.median(spectrum))

ax.set_xlabel("Frequency (MHz)")
ax.set_ylabel("Power (dB)")
ax.set_title("Autocorrelation power spectra (Sun-transit median over time)")
ax.grid(alpha=0.3)
ax.legend(fontsize=6, ncol=4, loc='lower center')
fig.tight_layout()
plt.show()

print("Per-antenna median |V_ii| (mid-band):")
arr = np.array(list(auto_power_median.values()))
print(f"  median across antennas: {np.median(arr):.3g}")
print(f"  ratio to median:")
for aid, p in sorted(auto_power_median.items(), key=lambda x: x[1]):
    ratio = p / np.median(arr)
    flag = "⚠️ low" if ratio < 0.3 else ""
    print(f"    ant {aid:>3}: {p:.3g}  ({ratio:.2f}×)  {flag}")

## 7. Waterfall visualization

The waterfall matrix shows |V_ij| or arg(V_ij) for every active baseline as
a 2D time-frequency panel. Useful for spotting RFI, fringe patterns, and
bandpass shape. The diagonal shows autocorrelations; the upper triangle shows
cross-correlation phase.

This is also where you'd visually check Sun's fringes during transit —
they should appear as smooth diagonal stripes in the phase plot.

In [ ]:
from casm_vis_analysis.plotting.waterfall import plot_waterfall

# Build the per-antenna labels and SNAP-ADC labels for titles
pkt_indices = [ant.packet_index(a) for a in aids]
snap_adc = [ant.snap_adc(a) for a in aids]
snap_adc_labels = [f"S{s}A{a}" for s, a in snap_adc]
antenna_labels = [f"ant{aid}\n{snap_adc_labels[i]}" for i, aid in enumerate(aids)]

# Plot — split into groups of 8 antennas to keep panels readable
figs = plot_waterfall(
    sun_data.vis, sun_data.freq_mhz, sun_data.time_unix,
    nsig=nsig,
    packet_indices=pkt_indices,
    antenna_labels=antenna_labels,
    snap_adc_labels=snap_adc_labels,
    split_max=8,                      # 8x8 grid per figure
    output_dir=None,                  # show inline
    diag_spectra=False,
    median_recipe=False,              # set True for complex-median subtract
)
for f in figs:
    plt.figure(f.number)
    plt.show()

## 8. Source ephemeris — Sun, Cyg A, Cas A

Where are the bright sources during our observation window? Use this to:
- Verify Sun's alt-az during cal solve (should be > 50° for good gain)
- Plan target observations when other sources won't contaminate via cross-talk
- Find the transit time for any target

`source_altaz(name, time_unix)` returns alt/az in degrees. Available names:
`sun`, `cyg_a`, `cas_a`, `tau_a`, `b0329_54`.

In [ ]:
# Build a time grid spanning the Sun transit window
t_grid = sun_data.time_unix
times_dt = [datetime.fromtimestamp(t, tz=ZoneInfo(TZ)) for t in t_grid]

fig, ax = plt.subplots(figsize=(9, 4.5))
for src, color in [("sun", "C1"), ("cyg_a", "C0"), ("cas_a", "C2"),
                   ("tau_a", "C3")]:
    alt, az = source_altaz(src, t_grid)
    ax.plot(times_dt, alt, color=color, label=f"{src} (max az={int(az.max())}°)",
            linewidth=1.4)

ax.axhline(0, color="0.5", linewidth=0.5, linestyle=":")
ax.axhline(15, color="r", linewidth=0.5, linestyle="--", label="alt = 15° threshold")
ax.set_ylabel("Altitude (deg)")
ax.set_xlabel(f"Local time ({TZ})")
ax.set_title(f"Source altitude during Sun-transit window")
ax.legend(loc="best", fontsize=9)
ax.grid(alpha=0.3)
from matplotlib.dates import DateFormatter
ax.xaxis.set_major_formatter(DateFormatter("%H:%M", tz=ZoneInfo(TZ)))
fig.autofmt_xdate()
fig.tight_layout()
plt.show()

# Quick transit window finder for any source within this obs
print("Transit windows (alt > 30°) within this obs:")
for src in ["sun", "cyg_a", "cas_a", "tau_a"]:
    try:
        i_start, i_end = find_transit_window(src, t_grid, min_alt_deg=30.0)
        if i_end > i_start:
            t0 = times_dt[i_start].strftime("%H:%M")
            t1 = times_dt[i_end-1].strftime("%H:%M") if i_end > 0 else "—"
            alt, _ = source_altaz(src, t_grid[i_start:i_end])
            print(f"  {src:>6}: {t0} → {t1}  peak alt={alt.max():.1f}°")
    except Exception as e:
        print(f"  {src:>6}: never above 30°  ({e})")

## 9. Static visibility subtraction

CASM's compact array has measurable cross-talk between adjacent antennas
(ε ≈ 2–35% for |b| < 5 m per the CASM-256 paper). This shows up as
time-stationary contributions in V_ij that aren't from any sky source.

The "static" cache is a vis snapshot taken when all bright sources are well
below horizon — it captures cross-talk + instrumental DC offsets. Subtracting
it from science data cleans up most baseline-dependent DC bias.

For full-baseline imaging this isn't strictly required (the dirty-image
geometry handles DC well), but it makes the per-baseline waterfall plots
much cleaner.

In [ ]:
static = load_static_visibility(STATIC_NPZ)
print(f"  static_vis shape : {static['static_vis'].shape}")
print(f"  freq_mhz range   : {static['freq_mhz'][0]:.2f} → {static['freq_mhz'][-1]:.2f} MHz")

# Sanity check that frequency axes match before subtracting
assert np.allclose(static['freq_mhz'], sun_data.freq_mhz), \
    "static and data freq axes don't match — rebuild the static for this obs?"

sun_data_subtracted = subtract_static_visibility(sun_data, static['static_vis'])
print(f"\nStatic subtracted. vis shape unchanged: {sun_data_subtracted['vis'].shape}")

## 10. Fringe-stop on Sun

Each baseline sees Sun's natural fringe rotation: V_ij ∝ exp(+2πi f τ_geom),
where τ_geom = b·ŝ / c. Fringe-stopping multiplies by exp(−2πi f τ_geom) so
that V_ij becomes constant in time at Sun's direction. After fringe-stopping,
the SVD is solving for "what gain pattern G_i makes all baselines look like
the same complex number" — that's a clean rank-1 problem.

**Sign convention**: `sign=-1` (default) removes the natural fringe phase.
Don't change this — it's what every downstream stage expects.

In [ ]:
fs = fringe_stop(sun_data_subtracted, ant, ref_ant=REF_ANT, source="sun", sign=-1)

print(f"  vis_stopped shape : {fs['vis_stopped'].shape}")
print(f"  ref_ant           : {fs['ref_ant']}  (packet index = {pkts[aids.index(REF_ANT)]})")
print(f"  target_aids       : {fs['target_aids']}")
print(f"  time_mask True    : {int(fs['time_mask'].sum())} / {len(fs['time_mask'])}")
print(f"\nThe stopped vis should look ~constant in time at all baselines —")
print(f"any time-variation indicates per-antenna gain phase or noise.")

## 11. SVD calibration

Per-channel SVD on the fringe-stopped vis matrix. Two modes:

- **`PHASE_ONLY`** (production default, threshold=5): keeps only phase per
  antenna per channel, |g| = 1. Throws away bandpass amplitude information
  but stays well-conditioned even at modest signal-to-noise.
- **`COMPLEX`** (validated 2026-05-15, threshold=4): keeps both amplitude and
  phase, |g| varies. Recovers ~15% in image peak amplitude but the SVD's
  rank-1 fit is harder, so fewer channels pass the threshold.

`fill_failed="zero"` means bad channels get gain = 0 → effectively dropped
from downstream imaging.

In [ ]:
# Phase-only cal (safest production default)
cfg_phase = SVDConfig(
    threshold=5.0,
    svd_mode=SVDMode.PHASE_ONLY,
    block_size=1,                # per-channel SVD
    subband_size=None,
    fill_failed="zero",
    masked_band_strategy="zero",
    smooth_order=0,
)
cal_phase = svd_calibrate(fs, ant, data=sun_data_subtracted, config=cfg_phase)
save_calibration(cal_phase, str(OUT_DIR / "cal" / "cal_phase.h5"),
                 n_time_averaged=fs['vis_stopped'].shape[0])

good_phase = int(cal_phase['flags'].sum())
print(f"PHASE_ONLY cal:  good_ch = {good_phase} / {len(cal_phase['flags'])}  "
      f"(rank1 median = {float(np.median(cal_phase['rank1_ratios'][cal_phase['flags']])):.2f})")

In [ ]:
# COMPLEX cal (amp + phase, threshold=4 recovers bandpass amplitude)
cfg_complex = SVDConfig(
    threshold=4.0,
    svd_mode=SVDMode.COMPLEX,
    block_size=1,
    subband_size=None,
    fill_failed="zero",
    masked_band_strategy="zero",
    smooth_order=0,
)
cal_complex = svd_calibrate(fs, ant, data=sun_data_subtracted, config=cfg_complex)
save_calibration(cal_complex, str(OUT_DIR / "cal" / "cal_complex.h5"),
                 n_time_averaged=fs['vis_stopped'].shape[0])

good_complex = int(cal_complex['flags'].sum())
print(f"COMPLEX cal:     good_ch = {good_complex} / {len(cal_complex['flags'])}  "
      f"(rank1 median = {float(np.median(cal_complex['rank1_ratios'][cal_complex['flags']])):.2f})")

# Choose which cal to use downstream. Use phase-only as the safe default.
cal = cal_phase   # or cal_complex
print(f"\nUsing: PHASE_ONLY cal with {good_phase} good channels")

## 12. Inspect cal solution

Look at the per-antenna gain phase vs frequency. Sanity checks:
- Phases should be SMOOTH (no random walk → real per-antenna gain)
- A linear ramp across frequency = cable delay (you can extract it via
  `casm_vis_analysis.delay.fit_delay`)
- Sharp jumps at sub-band boundaries usually indicate the SVD stitching is
  finding spurious channel groupings — check `subband_size`

If `cal_complex` was used: also look at |g| per channel to see the bandpass
shape each antenna sees.

In [ ]:
# Plot per-antenna gain phase vs freq
n_ant = len(aids)
gains = cal['gains']                 # (n_ant, n_chan) complex
freq_mhz_cal = sun_data.freq_mhz
good_mask = cal['flags']

fig, axes = plt.subplots(7, 3, figsize=(11, 12), sharex=True, sharey=True)
for k, ax in enumerate(axes.ravel()):
    if k >= n_ant:
        ax.set_visible(False); continue
    phase_deg = np.degrees(np.angle(gains[k]))
    # Mask out cal-failed channels for cleanliness
    phase_deg_masked = np.where(good_mask, phase_deg, np.nan)
    ax.plot(freq_mhz_cal, phase_deg_masked, linewidth=0.5)
    ax.set_title(f"ant {aids[k]}", fontsize=8)
    ax.grid(alpha=0.3)
fig.suptitle("Per-antenna gain phase vs frequency (cal-flagged channels only)")
fig.supxlabel("Freq (MHz)"); fig.supylabel("Phase (deg)")
fig.tight_layout()
plt.show()

## 13. Full upper-triangle imaging — Sun closure sanity

Apply cal to vis and image at Sun direction. This is the **closure sanity
check**: applying Sun-derived cal to Sun-transit data and imaging at Sun's
position should give a clean centered peak at amplitude ≈ 1 (after bandpass
normalization).

Key recipe:
- Loop over ALL `(i, j)` pairs with `i < j` (full upper-triangle)
- Filter to `|b_ij| > 5 m` (cross-talk avoidance, paper Sec 8.1)
- Apply per-baseline cal: `V_ij_cal = V_ij × conj(g_i × conj(g_j))`
- Bandpass-normalize: divide each (channel, baseline) by `median_t |V|`
- Frequency-average to ~30 frequency chunks for speed
- Pass to `make_altaz_image` with time-varying source pointing

In [ ]:
BASELINE_THRESHOLD_M = 5.0          # cross-talk cutoff (production default)
N_FREQ_AVG = 32                     # freq channels per chunk

def image_full_baselines(vis, freq_mhz, time_unix, ant, cal, target_name,
                          ctrl_offset_deg=0, ang_max_deg=15.0, npix=81,
                          baseline_threshold_m=BASELINE_THRESHOLD_M):
    """Full upper-tri imager. Returns (image, daz, dalt, n_pairs).

    baseline_threshold_m: drop baselines with |b| <= this many meters.
        Default = 5 m (cross-talk avoidance for compact sources).
        For RESOLVED sources like the Sun at 437 MHz, pass 0.0 — the long
        baselines partly resolve the corona and the short ones carry the
        bulk flux.
    """
    aids = ant.active_antennas()
    pkts = np.array([ant.packet_index(a) for a in aids])
    positions = ant.get_positions()
    pos_active = positions[pkts]
    n_ant = len(aids)
    nsig = int(fmt.nsig)

    # Enumerate upper-tri pairs with |b| > threshold
    pairs = []
    for i_loc in range(n_ant):
        for j_loc in range(i_loc + 1, n_ant):
            b = pos_active[j_loc] - pos_active[i_loc]
            if np.linalg.norm(b) > baseline_threshold_m:
                pairs.append((i_loc, j_loc, b))
    n_pairs = len(pairs)
    baselines = np.array([p[2] for p in pairs])

    # Filter to cal-flagged good channels
    good_chans = np.where(cal['flags'])[0]
    cal_g_good = cal['gains'][:, good_chans]
    freq_mhz_good = freq_mhz[good_chans]
    T = vis.shape[0]

    # Build calibrated (T, F_good, n_pairs) array
    bl_vis = np.empty((T, len(good_chans), n_pairs), dtype=np.complex64)
    for k, (i_loc, j_loc, b) in enumerate(pairs):
        pi, pj = pkts[i_loc], pkts[j_loc]
        lo, hi = min(pi, pj), max(pi, pj)
        v = vis[:, good_chans, triu_flat_index(nsig, lo, hi)]
        if pi > pj:
            v = np.conj(v)
        cal_pair = cal_g_good[i_loc] * np.conj(cal_g_good[j_loc])
        bl_vis[:, :, k] = (v * np.conj(cal_pair)).astype(np.complex64)

    # Bandpass-normalize
    bp = np.nanmedian(np.abs(bl_vis), axis=0, keepdims=True)
    bp = np.where(bp > 0, bp, 1.0)
    bl_vis = bl_vis / bp

    # Freq-average
    F_use = (len(good_chans) // N_FREQ_AVG) * N_FREQ_AVG
    bl_vis = bl_vis[:, :F_use, :].reshape(
        T, F_use // N_FREQ_AVG, N_FREQ_AVG, n_pairs).mean(axis=2)
    freq_use = freq_mhz_good[:F_use].reshape(F_use // N_FREQ_AVG, N_FREQ_AVG).mean(axis=1)
    freq_hz_use = freq_use * 1e6

    # Source position (time-varying for moving sources like Sun)
    alt_deg, az_deg = source_altaz(target_name, time_unix)
    if ctrl_offset_deg != 0:
        # Control direction = offset in az from source mean
        mean_alt = float(np.mean(alt_deg)); mean_az = float(np.mean(az_deg))
        alt_deg = np.full(T, mean_alt)
        az_deg = np.full(T, (mean_az + ctrl_offset_deg) % 360)

    image, daz, dalt = make_altaz_image(
        bl_vis, freq_hz_use, baselines,
        np.deg2rad(alt_deg), np.deg2rad(az_deg),
        ang_max_deg=ang_max_deg, npix=npix, fs_sign=-1,
        include_conjugates=False, verbose=False,
    )
    return image, daz, dalt, n_pairs

# === Sun closure: apply Sun-cal to Sun-transit, image at Sun ===
# Pass baseline_threshold_m=0.0 — the Sun is RESOLVED at 437 MHz (corona
# ~15-30 arcmin) so |b|>5m drops the short baselines that carry its bulk
# flux. Long-baseline-only imaging of a resolved source = Fourier high-pass
# of the brightness distribution = blob, not a peak. The production cutoff
# stays at 5 m for compact sources like Cyg A.
img_sun, daz_sun, dalt_sun, n_pairs_sun = image_full_baselines(
    sun_data_subtracted['vis'], sun_data_subtracted['freq_mhz'],
    sun_data_subtracted['time_unix'],
    ant, cal, target_name="sun",
    baseline_threshold_m=0.0,
)

# Plot
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.pcolormesh(daz_sun, dalt_sun, np.abs(img_sun), cmap="viridis", shading="auto")
ax.set_xlabel("Δaz from Sun (deg)")
ax.set_ylabel("Δalt from Sun (deg)")
ax.set_title(f"Sun-cal applied to Sun (closure)\nn_baselines={n_pairs_sun},  peak={np.abs(img_sun).max():.3g}")
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
plt.show()

print(f"\nPeak amplitude near (0, 0): {np.abs(img_sun)[40, 40]:.3g}")
print("Sanity: should be ≈ 1 if the cal closes properly on its own source.")

## 14. Cyg A image — cal transfer to a target

Apply the Sun cal to a DIFFERENT time window observing Cyg A. The image
should show Cyg A at its predicted position within the array PSF.

**Critical**: this only works cleanly when Sun is BELOW horizon during the
Cyg A window. Sun-up windows need the Sun-tracked beam subtraction we'll
show in the next section.

In [ ]:
# Read Cyg A vis
cyga_data = read_visibilities(
    time_start=CYGA_START, time_end=CYGA_END, time_tz=TZ,
    data_root=DATA_ROOT, fmt=fmt, verbose=False,
)
assert np.allclose(static['freq_mhz'], cyga_data.freq_mhz)
cyga_data = subtract_static_visibility(cyga_data, static['static_vis'])

# Sanity: confirm Sun is below horizon
sun_alt, _ = source_altaz("sun", cyga_data['time_unix'])
print(f"Sun alt during Cyg A window: {sun_alt.min():.1f}° → {sun_alt.max():.1f}°")
assert sun_alt.max() < -1, f"Sun is up during Cyg A window; use Sun-subtract recipe"

# Image at Cyg A direction (source-tracked)
img_cyga, daz, dalt, n_pairs = image_full_baselines(
    cyga_data['vis'], cyga_data['freq_mhz'], cyga_data['time_unix'],
    ant, cal, target_name="cyg_a",
)
# Also image at a control direction (30° az offset) to estimate noise
img_ctrl, _, _, _ = image_full_baselines(
    cyga_data['vis'], cyga_data['freq_mhz'], cyga_data['time_unix'],
    ant, cal, target_name="cyg_a", ctrl_offset_deg=30.0,
)

# Display side by side
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
peak_cyga = np.abs(img_cyga).max()
peak_ctrl = np.abs(img_ctrl).max()
for ax, img, title in [
    (axes[0], img_cyga, f"Cyg A direction (n_bl={n_pairs}, peak={peak_cyga:.3g})"),
    (axes[1], img_ctrl, f"Control direction +30° az (peak={peak_ctrl:.3g})"),
]:
    im = ax.pcolormesh(daz, dalt, np.abs(img), cmap="viridis", shading="auto")
    ax.set_xlabel("Δaz (deg)"); ax.set_ylabel("Δalt (deg)")
    ax.set_title(title)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.suptitle("Cal-to-target test: Sun-cal applied to Cyg A vs control")
fig.tight_layout()
plt.show()

print(f"\nSource/control peak ratio: {peak_cyga / peak_ctrl:.2f}×")
print("Above ~5× is a clear detection; above ~20× is unambiguous.")

## 14.5 Full-sky imaging — two methods

The `make_altaz_image` function we just used is a **tangent-plane projection**
centered at the target. It's correct only out to ~30° from the tangent point —
beyond that, the pixel labels don't map cleanly to real sky directions.

For a true **full-sky view** with sources at their actual alt-az positions,
we have two options:

1. **Snapshot**: form a single-integration beam at every pixel. No source
   motion (over 137 s the sky moves <1°), so sources appear as compact peaks
   with PSF fringes. Lower SNR per pixel (one integration vs 40).

2. **Celestial-tracked**: anchor each pixel to a fixed (RA, Dec). At each
   time, rotate the pixel's ENU vector by Earth's rotation since `t_mid`
   (Rodrigues' formula around the polar axis), then fringe-stop and sum
   coherently across all 40 integrations. Full SNR, sources stay sharp.

Both implemented as a per-pixel coherent beamformer (no `make_altaz_image`
machinery — pure phase × visibility sum).

In [ ]:
# Build the calibrated bandpass-normalized (T, F_avg, n_pairs) vis once
# (we use the same recipe as Section 13's image_full_baselines internals)
NPIX_PER_SIDE = 121          # ~1.5° per pixel; bump to 181 for ~1°/pix
C_LIGHT = 299792458.0

aids = ant.active_antennas()
pkts_idx = np.array([ant.packet_index(a) for a in aids])
positions = ant.get_positions()
pos_active = positions[pkts_idx]
n_ant = len(aids)
nsig = int(fmt.nsig)

pairs = []
for i_loc in range(n_ant):
    for j_loc in range(i_loc + 1, n_ant):
        b = pos_active[j_loc] - pos_active[i_loc]
        if np.linalg.norm(b) > 5.0:
            pairs.append((i_loc, j_loc, b))
baselines = np.array([p[2] for p in pairs])
n_pairs = len(pairs)

good_chans = np.where(cal['flags'])[0]
cal_g_good = cal['gains'][:, good_chans]
freq_mhz_good = cyga_data['freq_mhz'][good_chans]
T = cyga_data['vis'].shape[0]

# (T, F_good, n_pairs) calibrated and bandpass-normalized
bl_vis = np.empty((T, len(good_chans), n_pairs), dtype=np.complex64)
for k, (i_loc, j_loc, b) in enumerate(pairs):
    pi, pj = pkts_idx[i_loc], pkts_idx[j_loc]
    lo, hi = min(pi, pj), max(pi, pj)
    v = cyga_data['vis'][:, good_chans, triu_flat_index(nsig, lo, hi)]
    if pi > pj:
        v = np.conj(v)
    cal_pair = cal_g_good[i_loc] * np.conj(cal_g_good[j_loc])
    bl_vis[:, :, k] = (v * np.conj(cal_pair)).astype(np.complex64)
bp = np.nanmedian(np.abs(bl_vis), axis=0, keepdims=True)
bp = np.where(bp > 0, bp, 1.0)
bl_vis = bl_vis / bp
F_use = (len(good_chans) // 32) * 32
bl_vis = bl_vis[:, :F_use, :].reshape(T, F_use // 32, 32, n_pairs).mean(axis=2)
freq_hz = freq_mhz_good[:F_use].reshape(F_use // 32, 32).mean(axis=1) * 1e6
print(f"bl_vis shape: {bl_vis.shape}   freq_hz: {len(freq_hz)} chunks  baselines: {n_pairs}")

In [ ]:
# === METHOD 1: SNAPSHOT ===
# Single middle-integration beam at every alt-az pixel.

t_snap = T // 2
V_snap = bl_vis[t_snap]
t_snap_unix = cyga_data['time_unix'][t_snap]
print(f"snapshot at integration {t_snap}, t = "
      f"{datetime.fromtimestamp(t_snap_unix, tz=ZoneInfo(TZ)):%H:%M:%S}")

# Zenithal-equidistant grid: (x, y) in degrees from zenith
xs = np.linspace(-90, 90, NPIX_PER_SIDE)
ys = np.linspace(-90, 90, NPIX_PER_SIDE)
X, Y = np.meshgrid(xs, ys)
zd = np.sqrt(X**2 + Y**2)
above_horizon = zd <= 90.0
az_grid = (np.degrees(np.arctan2(X, Y))) % 360.0
alt_grid = np.where(above_horizon, 90.0 - zd, 0.0)
alt_rad = np.deg2rad(alt_grid); az_rad = np.deg2rad(az_grid)
s_mid = np.stack([np.sin(az_rad) * np.cos(alt_rad),
                   np.cos(az_rad) * np.cos(alt_rad),
                   np.sin(alt_rad)], axis=-1).reshape(-1, 3)

# tau per (pixel, baseline)
tau = (s_mid @ baselines.T) / C_LIGHT
print(f"  forming snapshot image ({s_mid.shape[0]} pixels)...")
image_flat = np.zeros(s_mid.shape[0], dtype=np.float32)
for p0 in range(0, s_mid.shape[0], 2048):
    p1 = min(p0 + 2048, s_mid.shape[0])
    ph = np.exp(-2j * np.pi * freq_hz[None, :, None] * tau[p0:p1, None, :])
    image_flat[p0:p1] = np.abs((V_snap[None, :, :] * ph).sum(axis=(1, 2)))
image_snap = image_flat.reshape(NPIX_PER_SIDE, NPIX_PER_SIDE)
image_snap = np.where(above_horizon, image_snap, np.nan)
print(f"  snapshot peak: {np.nanmax(image_snap):.3g}")

In [ ]:
# === METHOD 2: CELESTIAL-TRACKED ===
# Pixel anchored at (alt, az) at t_mid → fixed (RA, Dec) celestial direction.
# At each other time t, rotate that ENU vector around the polar axis by Earth's
# rotation since t_mid. Fringe-stop to the rotated direction at time t. Sum.

OVRO_LAT_DEG = 37.2314
OMEGA_EARTH = 2 * np.pi / 86164.0905   # sidereal rad/s
lat_rad = np.deg2rad(OVRO_LAT_DEG)
polar_axis = np.array([0.0, np.cos(lat_rad), np.sin(lat_rad)])

def rodrigues(s, axis, theta):
    cos_t = np.cos(theta); sin_t = np.sin(theta)
    cross = np.cross(np.broadcast_to(axis, s.shape), s)
    dot = (s @ axis)[..., None]
    return cos_t * s + sin_t * cross + (1 - cos_t) * dot * np.broadcast_to(axis, s.shape)

t_mid_unix = cyga_data['time_unix'][T // 2]
print(f"  forming celestial-tracked image ({s_mid.shape[0]} pix × {T} times)...")
image_acc = np.zeros(s_mid.shape[0], dtype=np.complex128)
for t_idx in range(T):
    dt = cyga_data['time_unix'][t_idx] - t_mid_unix
    theta = OMEGA_EARTH * dt
    s_t = rodrigues(s_mid, polar_axis, theta)
    tau_t = (s_t @ baselines.T) / C_LIGHT
    for p0 in range(0, s_mid.shape[0], 4096):
        p1 = min(p0 + 4096, s_mid.shape[0])
        ph = np.exp(-2j * np.pi * freq_hz[None, :, None] * tau_t[p0:p1, None, :])
        beam = (bl_vis[t_idx, None, :, :] * ph).sum(axis=(1, 2))
        image_acc[p0:p1] += beam
image_track = np.abs(image_acc).reshape(NPIX_PER_SIDE, NPIX_PER_SIDE)
image_track = np.where(above_horizon, image_track, np.nan)
print(f"  celestial-tracked peak: {np.nanmax(image_track):.3g}")

In [ ]:
# Plot the two full-sky images side-by-side
fig, axes = plt.subplots(1, 2, figsize=(14, 7.5))

# Predicted source positions at snapshot/t_mid
for name in ["cyg_a", "cas_a", "tau_a"]:
    alt_s, az_s = source_altaz(name, np.array([t_mid_unix]))
    if alt_s[0] < 0:
        continue
    zd_s = 90.0 - alt_s[0]
    xs_p = zd_s * np.sin(np.deg2rad(az_s[0]))
    ys_p = zd_s * np.cos(np.deg2rad(az_s[0]))
    for ax in axes:
        ax.plot(xs_p, ys_p, "o", markeredgecolor="red",
                markerfacecolor="none", markersize=12, markeredgewidth=1.5)
        ax.annotate(name.upper().replace("_", " "), (xs_p, ys_p),
                    textcoords="offset points", xytext=(8, 6),
                    fontsize=8, fontweight="bold", color="red")

for ax, img, label in [
    (axes[0], image_snap, "SNAPSHOT (single 137-s integration)"),
    (axes[1], image_track, "CELESTIAL-TRACKED (90-min coherent)"),
]:
    pcm = ax.pcolormesh(X, Y, img, cmap="viridis", shading="auto",
                        vmin=0, vmax=np.nanpercentile(img, 99.5))
    # horizon + alt rings
    th = np.linspace(0, 2 * np.pi, 200)
    for zr in (90, 60, 30):
        ax.plot(zr * np.sin(th), zr * np.cos(th),
                color="0.65", linewidth=0.4, linestyle="--")
    # NSEW
    for n_, x_, y_ in [("N", 0, 90), ("E", 90, 0), ("S", 0, -90), ("W", -90, 0)]:
        ax.text(x_ * 1.06, y_ * 1.06, n_, fontsize=11, fontweight="bold",
                ha="center", va="center",
                bbox=dict(boxstyle="circle,pad=0.3", facecolor="white", edgecolor="0.3"))
    ax.set_xlim(-95, 95); ax.set_ylim(-95, 95); ax.set_aspect("equal")
    ax.set_xlabel("E ← x (°) → W"); ax.set_ylabel("S ← y (°) → N")
    ax.set_title(label, fontsize=10)
    plt.colorbar(pcm, ax=ax, fraction=0.046, pad=0.04)
fig.suptitle("Full-sky CASM image: snapshot vs celestial-tracked",
             fontsize=12, fontweight="bold")
fig.tight_layout()
plt.show()

print("\nWhat to look for:")
print("  - SNAPSHOT: Cyg A and Cas A should appear at their alt-az positions")
print("    AT THE SNAPSHOT TIME, with PSF fringes radiating outward.")
print("  - CELESTIAL-TRACKED: same positions (anchored at t_mid), but with")
print("    full 90-min SNR → ~√40 ≈ 6× cleaner background.")

## 15. Sun-tracked beam subtract — for daytime windows

If your science window has Sun above horizon (e.g., daytime Cas A transit
or post-sunrise Cyg A descent), Sun contaminates every beam via PSF sidelobes
and short-baseline cross-talk.

The fix: form a beam pointed at Sun's tracked position over time, and subtract
it from each target beam.

`beam_power_vs_time(sources=["sun"])` accepts source NAMES directly and
internally tracks the time-varying alt/az. Gate the subtraction to
`Sun_alt > 0°` so it does nothing when Sun is below horizon.

In [ ]:
# Example: a window covering sunrise where Sun rises into the beams
WINDOW_START = "2026-05-14 04:00:00"
WINDOW_END   = "2026-05-14 07:30:00"   # ends 1h45m after sunrise

# Read a window that has Sun rising in it
sunrise_data = read_visibilities(
    time_start=WINDOW_START, time_end=WINDOW_END, time_tz=TZ,
    data_root=DATA_ROOT, fmt=fmt, verbose=False,
)
sunrise_data = subtract_static_visibility(sunrise_data, static['static_vis'])
print(f"Window covers {WINDOW_START} → {WINDOW_END}, {sunrise_data.vis.shape[0]} ints")

# Form the Sun-tracked beam + an example Cyg A-direction beam
bp = beam_power_vs_time(
    sunrise_data, ant,
    sources=["sun", "cyg_a"],     # accepts named sources + tuples
    cal_weights=load_calibration_weights(str(OUT_DIR / "cal" / "cal_phase.h5")),
    freq_band_mhz=None,
)
t_unix = bp['time_unix']
sun_power = np.asarray(bp['power']['sun'])
cyga_power = np.asarray(bp['power']['cyg_a'])

# Gate subtraction to when Sun is above horizon
sun_alt, _ = source_altaz("sun", t_unix)
gate = sun_alt > 0

# Use a 30-sample pre-rise window to estimate the Sun-beam DC offset
if gate.any():
    pre_rise = ~gate
    sun_dc = float(np.median(sun_power[pre_rise][-30:])) if pre_rise.sum() > 0 else 0.0
    sun_contribution = np.where(gate, sun_power - sun_dc, 0.0)
else:
    sun_contribution = np.zeros_like(sun_power)

cyga_sub = cyga_power - sun_contribution

# Plot raw vs subtracted
times_dt = [datetime.fromtimestamp(t, tz=ZoneInfo(TZ)) for t in t_unix]
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].plot(times_dt, cyga_power, color="0.15", label="Cyg A beam (raw)")
axes[0].plot(times_dt, sun_power, color="C1", linewidth=0.8, label="Sun beam")
axes[0].axvline(times_dt[np.argmax(sun_alt > 0)], color="r", linestyle="--",
                 label="sunrise")
axes[0].set_ylabel("power"); axes[0].grid(alpha=0.3); axes[0].legend()
axes[0].set_title("Raw beam powers")

axes[1].plot(times_dt, cyga_sub, color="C0", linewidth=1.4,
              label="Cyg A − Sun (cleaned)")
axes[1].axhline(0, color="0.5", linewidth=0.5, linestyle=":")
axes[1].set_ylabel("power - Sun"); axes[1].grid(alpha=0.3); axes[1].legend()
axes[1].set_title("Cyg A after Sun-tracked subtraction")
axes[1].set_xlabel(f"Local time ({TZ})")
fig.tight_layout()
plt.show()

## 16. validate_source — beam grid validation

For a deployed int8 weights file (a set of beam pointings written for the
online beamformer), `validate_source` identifies which beams the source
crosses during the obs and forms per-beam power-vs-time plots with
pass/fail thresholds.

This is the canonical "did my cal+weights work" diagnostic.

In [ ]:
# Path to a deployed int8 weights file (e.g., the production May 11 one)
INT8_PATH = "/home/casm/scratch/bf_calibration_march2026/outputs/weights_R2_thr5_128beams_int8.h5"

if Path(INT8_PATH).exists():
    result = validate_source(
        int8_h5=INT8_PATH,
        data=cyga_data, ant=ant,
        cal_weights=load_calibration_weights(str(OUT_DIR / "cal" / "cal_phase.h5")),
        source="cyg-a",
        freq_band_mhz=None,
        n_control_beams=2,
        fwhm_factor=1.0,
        pass_ratio=5.0,
    )
    print_source_validation_summary(result)

    fig = plot_source_validation(result, time_tz=TZ)
    plt.show()
else:
    print(f"INT8 file not found at {INT8_PATH} — skipping. Generate one in section 17.")

## 17. Generate 512-beam int8 weights file

For the online voltage beamformer, we write a single h5 file containing the
per-(beam, channel, antenna, polarization) complex weight, quantized to int8.

Pipeline:
1. Generate a beam grid covering the primary beam (alt-az grid).
2. For each beam pointing, compute geometric phase per antenna per channel.
3. Multiply by the calibration weights to get the full deployed weight.
4. Quantize to int8 with `scale_factor=127`.
5. Save with metadata (antenna positions, frequencies, pointings).

In [ ]:
# Build a 512-beam grid covering the primary beam
# (16 az bins × 32 alt bins ≈ 512; tweak to fit your need)
pointings = generate_beam_grid_altaz(
    alt_min_deg=20.0, alt_max_deg=90.0,
    spacing_deg=3.0,                  # ~ matches the array PSF FWHM
)
print(f"generated {len(pointings)} pointings")

# Cap at 512 (production constraint)
if len(pointings) > 512:
    # Sample uniformly
    idx = np.linspace(0, len(pointings) - 1, 512, dtype=int)
    pointings = [pointings[i] for i in idx]
print(f"using {len(pointings)} pointings")

# Build int8 weights
positions = ant.get_positions()
active_mask = np.zeros(64, dtype=bool)
for a in aids:
    active_mask[ant.packet_index(a)] = True

cal_for_int8 = load_calibration_weights(str(OUT_DIR / "cal" / "cal_phase.h5"))
int8 = Int8StationaryWeights.build(
    pointings=pointings,
    antenna_positions=positions,
    active_mask=active_mask,
    cal_weights=cal_for_int8,
    scale_factor=127.0,
)
print(f"int8 weights array shape: {int8.weights_int8.shape}")
print(f"  (n_part_real/imag, n_chan, n_pol, n_beam, n_input) = "
      f"{int8.weights_int8.shape}")

# Save to disk
INT8_OUT = OUT_DIR / "weights" / "weights_notebook_int8.h5"
save_int8_weights_hdf5(int8, str(INT8_OUT))
print(f"\nwrote {INT8_OUT}  ({INT8_OUT.stat().st_size / 1e6:.1f} MB)")

## 18. Next steps — pulsars

You now have:
- ✅ A validated phase-only (or COMPLEX) cal saved to disk
- ✅ A 512-beam int8 weights file ready for the online voltage beamformer
- ✅ A working full-baseline imaging pipeline
- ✅ Sun-tracked beam subtraction for daytime targets

For pulsar science (B0329+54, Crab):
1. **Pick the season carefully.** May 2026 has B0329 transiting at noon
   (Sun nearby) — hopeless. November 2026: B0329 at alt 73° at midnight.
2. **Stream voltage data**, not visibilities. The online beamformer applies
   your int8 weights to voltage samples at 2 ms cadence → filterbank.
3. **Coherent dedispersion** before folding helps with within-channel DM
   smearing (especially for Crab at our band).
4. **Use a control beam pointing off-source** to discriminate real pulses
   from RFI that happens to be at the right DM/period.

The validated 21-antenna pipeline gives ~5–15σ folded B0329 detection at
moderate altitude tonight; bumps to ~30–100σ at zenith in November. As
CASM-256 fills out, all numbers scale by N (linear in antenna count for
folded SNR).